In [ ]:
from osgeo import gdal
import xarray as xr
import numpy as np
import rioxarray
import os
from datetime

In [43]:
netcdf_file_path = r"E:\work\climate_downscaling\Data\Raw\har_datasets\nc\HARv2_d10km_h_2d_prcp_2020.nc"
variable_name = "prcp"

In [44]:
ds =xr.open_dataset(netcdf_file_path)

In [45]:
ds = ds.rio.set_spatial_dims(x_dim="west_east", y_dim="south_north", inplace=True)

In [46]:
ds

<xarray.Dataset> Size: 3GB
Dimensions:      (time: 8784, south_north: 252, west_east: 381)
Coordinates:
  * time         (time) datetime64[ns] 70kB 2020-01-01T01:00:00 ... 2021-01-01
  * south_north  (south_north) float32 1kB -7.45e+05 -7.35e+05 ... 1.765e+06
  * west_east    (west_east) float32 2kB -1.675e+06 -1.665e+06 ... 2.125e+06
    lon          (south_north, west_east) float32 384kB ...
    lat          (south_north, west_east) float32 384kB ...
Data variables:
    prcp         (time, south_north, west_east) float32 3GB ...
Attributes: (12/33)
    TITLE:                HAR v2 d10km
    DATA_NOTES:           File generated with the output of successive model ...
    WRF_VERSION:           OUTPUT FROM WRF V4.1 MODEL
    CREATED_BY:           Xun Wang - xun.wang@tu-berlin.de
    INSTITUTION:          Technische Universität Berlin, Institut für Ökologi...
    CREATION_DATE:        18.02.2021 08:58:45
    ...                   ...
    GRID_X00:             -1.675001e+06
    GRID_Y00:             -744998.94
    GRID_X01:             -1.675001e+06
    GRID_Y01:             1.765001e+06
    GRID_NX:              381
    GRID_NY:              252

In [ ]:
raster_crs = (
    f"+proj=lcc "
    f"+lat_1={float(ds.PROJ_STANDARD_PAR1.strip())} "
    f"+lat_2={float(ds.PROJ_STANDARD_PAR2.strip())} "
    f"+lat_0={float(ds.PROJ_CENTRAL_LAT.strip())} "
    f"+lon_0={float(ds.PROJ_CENTRAL_LON.strip())} "
    f"+x_0={float(ds.PROJ_FALSE_EASTING.strip())} "
    f"+y_0={float(ds.PROJ_FALSE_NORTHING.strip())} "
    f"+a={float(ds.PROJ_SEMIMAJOR_AXIS.strip())} "
    f"+b={float(ds.PROJ_SEMIMINOR_AXIS.strip())} "
    f"+units=m +no_defs"
)

In [47]:
ds = ds.rio.write_crs(raster_crs, inplace=True) # Example for WGS84 (lat/lon)


In [49]:
ds_reprojected = ds.rio.reproject("EPSG:4326")

In [51]:
ds_reprojected.rio.crs

CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]')

In [52]:
output_dir = r"E:\work\climate_downscaling\Data\Raw\har_datasets\delete"

In [ ]:
nepal_time_offset = np.timedelta64(6, 'h')

for i, t in enumerate(ds.time.values):
    original_time = np.datetime64(t)
    nepal_time = original_time + nepal_time_offset
    t_str = str(nepal_time)[:13].replace("-", "")
    out_path = os.path.join(output_dir, f"{variable_name}_{t_str}.tif")
    # Select the variable at this time step
    ds_reprojected[variable_name].isel(time=i).rio.to_raster(out_path)
    print(f"Saved {out_path}")

Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T01.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T02.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T03.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T04.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T05.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T06.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T07.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T08.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T09.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T10.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T11.tif
Saved E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101

In [88]:
ds_reprojected["time"].isel(time = 0) + nepal_time

<xarray.DataArray 'time' ()> Size: 8B
array('2020-01-01T07:00:00.000000000', dtype='datetime64[ns]')
Coordinates:
    time         datetime64[ns] 8B 2020-01-01T01:00:00
    spatial_ref  int64 8B 0
Attributes:
    long_name:  Time

In [78]:
dir_path = r"E:\work\climate_downscaling\Data\Raw\har_datasets\delete"
file_list = os.listdir(dir_path)
tif_list = [file for file in file_list if file.endswith(".tif")]

In [79]:
import numpy as np
import rasterio

# Read first file and get nodata value
with rasterio.open(os.path.join(dir_path, tif_list[0])) as s0:
    profile = s0.profile
    nodata = profile.get('nodata', None)
    # print(nodata)
    first_arr = s0.read(1).astype(np.float32)
    
    # Create masked array if nodata exists
    if nodata is not None:
        sum_arr = np.ma.masked_equal(first_arr, nodata)
    else:
        sum_arr = first_arr

# Sum remaining files
for raster in tif_list[1:]:
    full_raster_path = os.path.join(dir_path, raster)
    print(full_raster_path)
    with rasterio.open(full_raster_path) as s:
        arr = s.read(1).astype(np.float32)
        
        # Mask nodata values
        if nodata is not None:
            arr = np.ma.masked_equal(arr, nodata)
        
        sum_arr += arr

# Convert back to regular array with nodata filled
if nodata is not None:
    sum_arr = sum_arr.filled(nodata)

# Write output
with rasterio.open(r"E:\work\climate_downscaling\Data\Raw\har_datasets\sum2020_hr_xarray.tif", 
                   "w", **profile) as dst:
    dst.write(sum_arr.astype(np.float32), 1)

nan
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T02.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T03.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T04.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T05.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T06.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T07.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T08.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T09.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T10.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T11.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T12.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prcp_20200101T13.tif
E:\work\climate_downscaling\Data\Raw\har_datasets\delete\prc